# CartoVec — Quickstart pédagogique

**Niveau** : 2e année géomatique (EABA Tunisie).

Ce notebook montre les **5 étapes** du pipeline simple, étape par étape, avec des images intermédiaires pour comprendre ce qui se passe.

1. Charger la carte
2. Recadrer (détecter le neatline)
3. Segmenter par couleur en HSV
4. Vectoriser les masques
5. Sauvegarder en GeoJSON

Pas de deep learning, pas d'agent — juste OpenCV + Shapely.

In [ ]:
# Imports + chemin vers le projet
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pipeline import preprocessing as prep
from pipeline import color_segmentation as colseg
from pipeline import vectorization as vec

print('Imports OK')

## Étape 1 — Charger la carte

OpenCV lit l'image en **BGR** (pas RGB). On la convertit en RGB juste pour l'affichage.

In [ ]:
INPUT = '../data/raw/carte.png'  # adapte ici si besoin

img_bgr = cv2.imread(INPUT)
assert img_bgr is not None, f'Carte introuvable : {INPUT}'
print(f'Dimensions : {img_bgr.shape[1]} x {img_bgr.shape[0]} px')

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title('Carte originale (BGR -> RGB pour affichage)')
plt.axis('off'); plt.show()

## Étape 2 — Recadrer sur le cadre cartographique

Le **neatline** est le cadre noir autour de la carte. La fonction `detect_map_frame()` le trouve par seuillage + détection de contours OpenCV.

On redimensionne d'abord à 2400 px max pour éviter d'utiliser trop de RAM.

In [ ]:
# 1) downscale
img_small = prep.downscale_if_too_large(img_bgr, max_dimension=2400)

# 2) detect frame
x1, y1, x2, y2 = prep.detect_map_frame(img_small)
print(f'Cadre detecte : x1={x1}, y1={y1}, x2={x2}, y2={y2}')

# 3) crop
img_crop = img_small[y1:y2, x1:x2].copy()
print(f'Image recadree : {img_crop.shape[1]} x {img_crop.shape[0]} px')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB))
axes[0].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                 fill=False, edgecolor='red', lw=2))
axes[0].set_title('Cadre detecte (rouge)')
axes[1].imshow(cv2.cvtColor(img_crop, cv2.COLOR_BGR2RGB))
axes[1].set_title('Apres recadrage')
for ax in axes: ax.axis('off')
plt.show()

## Étape 3 — Segmentation par couleur HSV

On convertit BGR → HSV puis on applique un **seuillage** pour chaque classe :

| Classe       | Couleur sur la carte     | Plage HSV |
|--------------|-------------------------|-----------|
| water        | Bleu (rivières, lacs)    | H 95–130  |
| vegetation   | Vert (forêts)            | H 35–95   |
| red_roads    | Rouge (routes)           | H 0–15 ou 155–180 |
| contours     | Marron (courbes)         | H 6–25    |
| buildings    | Gris foncé (bâtiments)   | V faible  |

In [ ]:
hsv = cv2.cvtColor(img_crop, cv2.COLOR_BGR2HSV)
layers = colseg.extract_all_color_layers(hsv)

print('Couches extraites :')
for name, mask in layers.items():
    print(f'  {name:12s} cover = {colseg.coverage_percent(mask):5.2f}%')

# Affichage des 5 masques cote a cote
n = len(layers)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
for ax, (name, mask) in zip(axes, layers.items()):
    ax.imshow(mask, cmap='gray')
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

## Étape 4 — Vectorisation

Chaque masque binaire est converti en **géométries Shapely** :

- Surfaces (eau, végétation, bâtiments) → **polygones** via `rasterio.features.shapes()`.
- Lignes (courbes, routes) → squelettisation puis **LineStrings**.

Le résultat est exporté en dict GeoJSON, lisible dans QGIS.

In [ ]:
geojsons = vec.masks_to_geojson(layers)

print('Features par couche :')
for name, gj in geojsons.items():
    print(f'  {name:12s} : {len(gj.get("features", []))} features')

## Étape 5 — Sauvegarder + ouvrir dans QGIS

On écrit chaque couche dans `data/processed/quickstart/<nom>.geojson`. Tu peux ensuite glisser-déposer ces fichiers dans QGIS pour les visualiser.

In [ ]:
import json
out_dir = Path('../data/processed/quickstart')
out_dir.mkdir(parents=True, exist_ok=True)

for name, gj in geojsons.items():
    out = out_dir / f'{name}.geojson'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(gj, f, indent=2)
    print(f'  -> {out}')

## Tout-en-un

Les 5 étapes ci-dessus sont encapsulées dans `pipeline.simple_pipeline.run_simple_pipeline()` :

In [ ]:
from pipeline.simple_pipeline import run_simple_pipeline
result = run_simple_pipeline(INPUT, '../data/processed/quickstart')
print(result)

## Pour aller plus loin (modules avancés)

Une fois la version simple comprise, tu peux explorer :

- `pipeline.semantic_segmentation` — U-Net deep learning (PyTorch)
- `pipeline.georeferencing` — GCPs + projection WGS84 (rasterio)
- `pipeline.agent` — orchestrateur LangGraph (Perceive → QA → Self-correct)
- `pipeline.active_learning` — corrections HSV adaptatives EMA

Tous ces modules **sont optionnels**. Le pipeline simple ci-dessus suffit déjà pour la démo PFA.